# Datamart check

Check if the same tables as A1's `python main.py` which built every month at one go are produced. 
A1 results: 12,500 applications in the feature store with 68 columns, 12,500 labels, a 1:1 join, and 28.8% bad.

In [1]:
import glob
import re
import pandas as pd

pd.set_option("display.width", 200)

## 1. Partitions and rows per layer

In [2]:
# one partition per month per table; months the source has no data for are skipped by data_processing.py
rows = []
for layer in ["bronze", "silver", "gold"]:
    for folder in sorted(glob.glob(f"../datamart/{layer}/*/")):
        table = folder.rstrip("/").split("/")[-1]
        partitions = sorted(glob.glob(folder + "*"))
        if layer == "bronze":
            n = sum(len(pd.read_csv(p)) for p in partitions)
        else:
            n = len(pd.read_parquet(folder))
        month = lambda p: re.search(r"\d{4}_\d{2}_\d{2}", p).group()
        rows.append([layer, table, len(partitions), month(partitions[0]), month(partitions[-1]), n])
pd.DataFrame(rows, columns=["layer", "table", "partitions", "first", "last", "rows"])

,layer,table,partitions,first,last,rows
0,bronze,attributes,25,2023_01_01,2025_01_01,12500
1,bronze,clickstream,24,2023_01_01,2024_12_01,215376
2,bronze,financials,25,2023_01_01,2025_01_01,12500
3,bronze,loan_daily,31,2023_01_01,2025_07_01,132419
4,silver,attributes,25,2023_01_01,2025_01_01,12500
5,silver,clickstream,24,2023_01_01,2024_12_01,215376
6,silver,financials,25,2023_01_01,2025_01_01,12500
7,silver,loan_daily,31,2023_01_01,2025_07_01,132419
8,gold,feature_store,25,2023_01_01,2025_01_01,12500
9,gold,label_store,31,2023_01_01,2025_07_01,12500


In [3]:
# bronze must equal the raw files for the months the DAG covered (up to its end date, 2025-07)
raw = {
    "attributes": "../data/features_attributes.csv",
    "financials": "../data/features_financials.csv",
    "clickstream": "../data/feature_clickstream.csv",
    "loan_daily": "../data/lms_loan_daily.csv",
}
for table, path in raw.items():
    df = pd.read_csv(path)
    in_window = (df["snapshot_date"] <= "2025-07-01").sum()
    bronze = sum(len(pd.read_csv(p)) for p in glob.glob(f"../datamart/bronze/{table}/*.csv"))
    print(f"{table:12s} raw rows up to 2025-07: {in_window:7d}  bronze rows: {bronze:7d}  equal: {in_window == bronze}")

attributes   raw rows up to 2025-07:   12500  bronze rows:   12500  equal: True
financials   raw rows up to 2025-07:   12500  bronze rows:   12500  equal: True
clickstream  raw rows up to 2025-07:  215376  bronze rows:  215376  equal: True
loan_daily   raw rows up to 2025-07:  132419  bronze rows:  132419  equal: True


clickstream  raw rows up to 2025-07:  215376  bronze rows:  215376  equal: True
loan_daily   raw rows up to 2025-07:  132419  bronze rows:  132419  equal: True


**Findings:**
- Every table has one partition per month that its source covers. Loan data runs to the DAG's end date (2025-07), so it has 31 partitions instead of A1's 35 (A1 ran to 2025-11). The four months after 2025-07 hold no label a 6-month label needs: the last applications (2025-01) mature in 2025-07.
- Bronze equals the raw files row for row in every table, for the months covered.

## 2. Gold stores check

In [4]:
features = pd.read_parquet("../datamart/gold/feature_store/")
labels = pd.read_parquet("../datamart/gold/label_store/")
print("feature store:", features.shape, "| duplicate keys:", features.duplicated(["Customer_ID", "snapshot_date"]).sum())
print("label store:", labels.shape, "| bad rate:", round(labels["label"].mean(), 3), "| label dates:", labels["snapshot_date"].min(), "to", labels["snapshot_date"].max())
joined = features.merge(labels, left_on=["Customer_ID", "snapshot_date"], right_on=["Customer_ID", "loan_start_date"], how="inner")
print("labels joined to exactly one application:", len(joined), "of", len(labels))

feature store: (12500, 68) | duplicate keys: 0
label store: (12500, 6) | bad rate: 0.288 | label dates: 2023-07-01 to 2025-07-01
labels joined to exactly one application: 12500 of 12500


In [5]:
# point in time: recompute one applicant's clickstream average from silver, using only months up to the application
clickstream = pd.read_parquet("../datamart/silver/clickstream/")
customer, applied = "CUS_0x10ee", features.loc[features["Customer_ID"] == "CUS_0x10ee", "snapshot_date"].iloc[0]
history = clickstream[(clickstream["Customer_ID"] == customer) & (clickstream["snapshot_date"] <= applied)]
print("applied:", applied, "| months used:", len(history), "| fe_1 recomputed:", round(history["fe_1"].mean(), 2),
      "| in the feature store:", round(features.loc[features["Customer_ID"] == customer, "fe_1_avg"].iloc[0], 2))

applied: 2023-06-01 | months used: 6 | fe_1 recomputed: 122.83 | in the feature store: 122.83


**Findings:**
- The feature store has 12,500 rows × 68 columns with no duplicate keys, and the label store 12,500 labels at 28.8% bad, dated 2023-07 to 2025-07. These are A1's numbers.
- All 12,500 labels join to exactly one application on customer + application month.
- CUS_0x10ee's `fe_1_avg` recomputed from silver (months up to the 2023-06 application) equals the stored value, 122.83, as in A1. Building month by month in Airflow kept the point-in-time rule.